In [ ]:
# Install:
# Kaggle environments.
!git clone https://github.com/Kaggle/kaggle-environments.git
!cd kaggle-environments && pip install .

# GFootball environment.
!apt-get update -y
!apt-get install -y libsdl2-gfx-dev libsdl2-ttf-dev

# Make sure that the Branch in git clone and in wget call matches !!
!git clone -b v2.8 https://github.com/google-research/football.git
!mkdir -p football/third_party/gfootball_engine/lib

!wget https://storage.googleapis.com/gfootball/prebuilt_gameplayfootball_v2.8.so -O football/third_party/gfootball_engine/lib/prebuilt_gameplayfootball.so
!cd football && GFOOTBALL_USE_PREBUILT_SO=1 pip3 install .

In [ ]:
# %%writefile submission.py

In [ ]:
%%writefile submission.py
from kaggle_environments.envs.football.helpers import *
from random import randint
import math

# Function to calculate distance
def get_distance(pos1,pos2):
    return (((pos1[0]-pos2[0])**2)+((pos1[1]-pos2[1])**2))**0.5

# Function to update snapShot
def set_snapShot(num):
    global snapShot
    snapShot = num
    return

# Function to get snapShot
def get_snapShot():
    global snapShot
    return snapShot

# Function to update snapShot
def set_gkPlay(num):
    global gkPlay
    gkPlay = num
    return

# Function to get snapShot
def get_gkPlay():
    global gkPlay
    return gkPlay


# Movement directions
directions = [
[Action.TopLeft, Action.Top, Action.TopRight],
[Action.Left, Action.Idle, Action.Right],
[Action.BottomLeft, Action.Bottom, Action.BottomRight]]

dirsign = lambda x: 1 if abs(x) < 0.01 else (0 if x < 0 else 2)

# Set game plan parameters
goalRange = 0.75
crossAngleRange = 0.9
wingRange = 0.2
wingPassRange = 0.12
wingBackRange = -0.15
snapShot = 0
gkPlay = 0


@human_readable_agent
def agent(obs):

    ################################
    # Functions and logical checks
    ################################

    # Functions

    # Add direction to action
    def sticky_check(action, direction, snap=False, gk=False):
        if direction in obs['sticky_actions']:
            if snap:
                set_snapShot(1)
            if gk:
                set_gkPlay(1)
            return action
        else:
            return direction

    # combo action patterns
    def combo_actions(action1, action2):
        if action1 in obs['sticky_actions']:
            return action2
        else:
            return action1

    # Sprint to dribble
    def sprint2dribble():
        if Action.Sprint in obs['sticky_actions']:
            return Action.ReleaseSprint
        else:
            return Action.Dribble

    # Dribble to sprint
    def dribble2sprint():
        if Action.Dribble in obs['sticky_actions']:
            return Action.ReleaseDribble
        else:
            return Action.Sprint

    # Sticky actions management
    def get_active_sticky_action(obs, exceptions):
#     """ get release action of the first active sticky action, except those in exceptions list """
        release_action = None
        for k in sticky_actions:
            if k not in exceptions and sticky_actions[k] in obs["sticky_actions"]:
                if k == "sprint":
                    release_action = Action.ReleaseSprint
                elif k == "dribble":
                    release_action = Action.ReleaseDribble
                else:
                    release_action = Action.ReleaseDirection
                break
        return release_action

    # Get coordinate distance
    def get_distance_4xy(x1, y1, x2, y2):
#     """ get two-dimensional Euclidean distance, considering y size of the field """
        return math.sqrt((x1 - x2) ** 2 + (y1 * 2.38 - y2 * 2.38) ** 2)

    # Check if player open
    def opp_prox(x, y, prox=0.05, front=False, back=False):
        xr = x + prox
        xl = x - prox
        yt = y - prox
        yb = y + prox

        if (x+prox) > 1:
            xr = 1
        if (x-prox) < -1:
            xl = -1
        if (y-prox) < -0.42:
            yt = -0.42
        if (y+prox) > 0.42:
            yb = 0.42

        cnt=0
        for opp in obs['right_team']:
            # If only checking opponents to the front
            if front:
                if opp[0] > x and opp[0] < xr and opp[1] > yt and opp[1] < yb:
                    cnt+=1
                    continue
            if back:
                if opp[0] < x and opp[0] > xl and opp[1] > yt and opp[1] < yb:
                    cnt+=1
                    continue
            # If checking in all directions
            if opp[0] > xl and opp[0] < xr and opp[1] > yt and opp[1] < yb:
                cnt+=1

        return cnt


    # Logical checks

    controlled_player_pos = obs['left_team'][obs['active']]

    if obs['ball_owned_team'] == 1 or abs(obs['ball'][1]) > wingRange or obs['game_mode'] != GameMode.Normal:
        set_snapShot(0)
    
    if abs(obs['ball'][1]) > wingPassRange:
        set_gkPlay(0)

    if get_snapShot() == 1:
        return Action.Shot
    
    if get_gkPlay() == 1:
        dirsign = lambda x: 1 if abs(x) < 0.01 else (0 if x < 0 else 2)
        # Run towards the ball.
        xdir = dirsign(obs['ball'][0] - controlled_player_pos[0])
        ydir = dirsign(obs['ball'][1] - controlled_player_pos[1])
        return directions[ydir][xdir]

    ###############
    # Game modes
    ###############

    # Pass when KickOff or FreeKick or ThrowIn
    if obs['game_mode'] == GameMode.KickOff or obs['game_mode'] == GameMode.ThrowIn:
        return sticky_check(Action.ShortPass, Action.Right)

    # Shoot when freekick in goal range; If on wing then cross; Otherwise just pass
    if obs['game_mode'] == GameMode.FreeKick:
        # Shoot if in range
        if controlled_player_pos[0] > goalRange and abs(controlled_player_pos[1]) < wingRange:
            ydir = randint(0,2)
            return sticky_check(Action.Shot, directions[ydir][2], snap=True)
        # Cross from right
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] > wingRange:
            if controlled_player_pos[0] > crossAngleRange:
                return sticky_check(Action.HighPass, Action.Top, snap=True)
            else:
                return sticky_check(Action.HighPass, Action.TopRight, snap=True)
        # Cross from left
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] < -(wingRange):
            if controlled_player_pos[0] > crossAngleRange:
                return sticky_check(Action.HighPass, Action.Bottom, snap=True)
            else:
                return sticky_check(Action.HighPass, Action.BottomRight, snap=True)
        return sticky_check(Action.HighPass, Action.Right)

    # Cross in for corner
    if obs['game_mode'] == GameMode.Corner and obs['ball'][1] < 0:
        return sticky_check(Action.HighPass, Action.Bottom, snap=True)
    elif obs['game_mode'] == GameMode.Corner and obs['ball'][1] > 0:
        return sticky_check(Action.HighPass, Action.Top, snap=True)

    # Long pass when GoalKick
    if obs['game_mode'] == GameMode.GoalKick:
        return sticky_check(Action.ShortPass, Action.Top, gk=True)

    # Shoot when Penalty
    if obs['game_mode'] == GameMode.Penalty:
        xdir = randint(0,2)
        ydir = randint(0,2)
        return sticky_check(Action.Shot, directions[ydir][xdir], snap=True)

    #####################
    # Sprint or Dribble
    #####################

    # Make sure player is running.
    # if Action.Sprint not in obs['sticky_actions']:
    #     return Action.Sprint


    # Make sure player is running.
    if  controlled_player_pos[0] < 0.6 and Action.Sprint not in obs['sticky_actions'] and opp_prox(controlled_player_pos[0], controlled_player_pos[1], prox=0.06, front=True) == 0:
        return dribble2sprint()
    elif 0.6 < controlled_player_pos[0] and Action.Dribble not in obs['sticky_actions']:
        if opp_prox(controlled_player_pos[0], controlled_player_pos[1], prox=0.03, back=True) == 0 and Action.Sprint in obs['sticky_actions']:
            return sprint2dribble()
        else:
            if opp_prox(controlled_player_pos[0], controlled_player_pos[1], prox=0.03, back=True) != 0 and opp_prox(controlled_player_pos[0], controlled_player_pos[1], prox=0.06, front=True) == 0 and Action.Sprint not in obs['sticky_actions']:
                return dribble2sprint()

    controlled_player_pos = obs['left_team'][obs['active']]

    ######################
    # Offensive play
    ######################

    # Check if we are in possession
    if obs['ball_owned_player'] == obs['active'] and obs['ball_owned_team'] == 0:

        # Defensive position
        # Clear if we are near our goal
        if controlled_player_pos[0] < -(goalRange) and abs(controlled_player_pos[1]) < wingPassRange:
            if opp_prox(controlled_player_pos[0], controlled_player_pos[1], prox=0.03, front=True) != 0:
                if controlled_player_pos[1] < 0:
                    return sticky_check(Action.LongPass, Action.Top)
                elif controlled_player_pos[1] > 0:
                    return sticky_check(Action.LongPass, Action.Bottom)
            return sticky_check(Action.Shot, Action.Right)


        # Wingback region
        if controlled_player_pos[0] < wingBackRange and abs(controlled_player_pos[1]) > wingPassRange:
            if opp_prox(controlled_player_pos[0], controlled_player_pos[1], front=True) == 0:
                # Launch down wing
                return sticky_check(Action.HighPass, Action.Right)
            else:
                return sticky_check(Action.Shot, Action.Right)


        # Offensive positions
        # Shoot if we are in the final third and not at an acute angle
        if controlled_player_pos[0] > goalRange and abs(controlled_player_pos[1]) < wingPassRange:
            set_snapShot(1)
            return Action.Shot
        # Shoot if goalie out of position
        elif (obs['right_team'][0][0] < 0.8 or abs(obs['right_team'][0][1]) > 0.1) and (controlled_player_pos[0] > 0.4) and abs(controlled_player_pos[1]) < wingRange:
            set_snapShot(1)
            return Action.Shot
        # Shoot if attacker very close to goalie
        # Player one on one with goalie
        goalkeeper_x = obs["right_team"][0][0] + obs["right_team_direction"][0][0] * 5
        goalkeeper_y = obs["right_team"][0][1] + obs["right_team_direction"][0][1] * 5
        # player have the ball and located close to the goalkeeper
        if get_distance_4xy(controlled_player_pos[0], controlled_player_pos[1], goalkeeper_x, goalkeeper_y) < 0.25:
            set_snapShot(1)
            return Action.Shot


        # Crossing positions
        # Cross from right
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] > wingRange:
            if controlled_player_pos[0] > crossAngleRange:
                return sticky_check(Action.HighPass, Action.Top, snap=True)
            else:
                return sticky_check(Action.HighPass, Action.TopRight, snap=True)

        # Pass from right
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] > wingPassRange and controlled_player_pos[1] < wingRange:
            if controlled_player_pos[0] > crossAngleRange:
                return sticky_check(Action.ShortPass, Action.Top, snap=True)
            else:
                return sticky_check(Action.ShortPass, Action.TopRight, snap=True)

        # Cross from left
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] < -(wingRange):
            if controlled_player_pos[0] > crossAngleRange:
                return sticky_check(Action.HighPass, Action.Bottom, snap=True)
            else:
                return sticky_check(Action.HighPass, Action.BottomRight, snap=True)

        # Pass from left
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] < -(wingPassRange) and controlled_player_pos[1] > -(wingRange):
            if controlled_player_pos[0] > crossAngleRange:
                return sticky_check(Action.ShortPass, Action.Bottom, snap=True)
            else:
                return sticky_check(Action.ShortPass, Action.BottomRight, snap=True)


        # Dribble if player nearby
        if opp_prox(controlled_player_pos[0], controlled_player_pos[1], prox=0.06, front=True) != 0:
            if Action.Sprint in obs['sticky_actions']:
                return sprint2dribble()
            # Pass if getting crowded
            if opp_prox(controlled_player_pos[0], controlled_player_pos[1], prox=0.02) != 0:
                team0s = [pos[0] for pos in obs['left_team']]
                team1s = [pos[1] for pos in obs['left_team']]
                if (controlled_player_pos[1] < 0 and controlled_player_pos[0] > 0 and controlled_player_pos[0] < goalRange and any(x > controlled_player_pos[0] and y < controlled_player_pos[1] for x,y in zip(team0s, team1s))) or \
                    (controlled_player_pos[1] < 0 and controlled_player_pos[0] < 0 and controlled_player_pos[0] > -(goalRange)):
                    return sticky_check(Action.ShortPass, Action.Top)

                if (controlled_player_pos[1] > 0 and controlled_player_pos[0] > 0 and controlled_player_pos[0] < goalRange and any(x > controlled_player_pos[0] and y > controlled_player_pos[1] for x,y in zip(team0s, team1s))) or \
                    (controlled_player_pos[1] > 0 and controlled_player_pos[0] < 0 and controlled_player_pos[0] > -(goalRange)):
                    return sticky_check(Action.ShortPass, Action.Bottom)


        # Stay within play area
        if controlled_player_pos[0] > 0.95:
            return Action.Left
        elif controlled_player_pos[1] > 0.4:
            return Action.Bottom
        elif controlled_player_pos[1] < -0.4:
            return Action.Top
        # Run towards the goal otherwise.
        return Action.Right


    ###################
    # Defensive play
    ###################

    else:

        opp_player_pos = obs['right_team'][obs['active']]
        opp_player_dir = obs['right_team_direction'][obs['active']]

        oppx = opp_player_pos[0] + opp_player_dir[0] * 7
        oppy = opp_player_pos[1] + opp_player_dir[1] * 7

        # Goalie charge
        team0s = [pos[0] for pos in obs['left_team']]
        if get_distance_4xy(oppx, oppy, obs['left_team'][0][0], obs['left_team'][0][1]) < 0.25 and \
            all(x > opp_player_pos[0] for x in team0s) and \
            obs['ball_owned_team'] == 1 and \
            obs['left_team'][0][0] < -(goalRange) and obs['game_mode'] == GameMode.Normal:
            return Action.LongPass

        #where ball is going we add the direction xy to ball current location
        ball_targetx=obs['ball'][0]+(5 * obs['ball_direction'][0])
        ball_targety=obs['ball'][1]+(5 * obs['ball_direction'][1])

#         # Euclidian distance to ball
#         e_dist=get_distance(obs['left_team'][obs['active']],obs['ball'])

#         if e_dist >.005 and obs['ball_owned_team'] == 1:
#             # Run where ball will be
#             xdir = dirsign(ball_targetx - controlled_player_pos[0])
#             ydir = dirsign(ball_targety - controlled_player_pos[1])
#             return directions[ydir][xdir]
#         else:
#             if obs['ball_owned_team'] == 1:
#                 return Action.ShortPass
# #             prob = randint(0,100)
# # #             if prob > 50 and controlled_player_pos[0] < obs['right_team'][obs['ball_owned_player']][0]:
# #             if prob >= 50 and obs['ball_owned_team'] == 1:
# #                 return Action.ShortPass
#             if obs['ball'][0] < goalRange:
#                 # Run towards the ball.
#                 xdir = dirsign(obs['ball'][0] - controlled_player_pos[0])
#                 ydir = dirsign(obs['ball'][1] - controlled_player_pos[1])
#                 return directions[ydir][xdir]
#             else:
#                 return Action.Idle

        dirsign = lambda x: 1 if abs(x) < 0.01 else (0 if x < 0 else 2)
        #where ball is going
        ball_targetx=obs['ball'][0]+(obs['ball_direction'][0]*5)
        ball_targety=obs['ball'][1]+(obs['ball_direction'][1]*5)

        e_dist=get_distance(obs['left_team'][obs['active']],obs['ball'])

        if e_dist >.01:
            # Run where ball will be
            xdir = dirsign(ball_targetx - controlled_player_pos[0])
            ydir = dirsign(ball_targety - controlled_player_pos[1])
            return directions[ydir][xdir]
        else:
            # Run towards the ball.
            xdir = dirsign(obs['ball'][0] - controlled_player_pos[0])
            ydir = dirsign(obs['ball'][1] - controlled_player_pos[1])
            return directions[ydir][xdir]


In [ ]:
# Set up the Environment.
from kaggle_environments import make
env = make("football", debug=True, configuration={"save_video": True, "scenario_name": "11_vs_11_kaggle", "running_in_notebook": True})
output = env.run(["/kaggle/working/submission.py", "run_right"])[-1]
print('Left player: reward = %s, status = %s, info = %s' % (output[0]['reward'], output[0]['status'], output[0]['info']))
print('Right player: reward = %s, status = %s, info = %s' % (output[1]['reward'], output[1]['status'], output[1]['info']))
env.render(mode="human", width=800, height=600)